# 📊 Dashboard de desempeño comercial 2024–2025
**Empresa:** Andes Retail Group (Perú, Chile y Colombia) · **Herramienta:** Power BI Desktop · **Autor:** Stefan Loosveld

Este notebook documenta el proceso completo del proyecto: exploración, preparación de datos, planificación del dashboard, construcción y narrativa SCQA. Las capturas del dashboard y el resumen de hallazgos están en el [README del repositorio](../README.md).

**Pregunta central:** ¿por qué cayeron los ingresos en 2025 y qué se puede hacer?

## 🔹 Paso 1: Conexión y exploración

**Objetivo:** importar el Excel en Power BI, revisar tipos de datos, identificar columnas clave y explorar la estructura.

**Lo que hice:**
- Conecté `Andes_Retail_Group_2024_2025.xlsx` con *Get data → Excel → Transform Data*, para revisar los datos en Power Query antes de cargarlos.
- Activé **Column quality** y **Column profile** con el perfil sobre **todo el dataset** (por defecto Power Query solo analiza las primeras 1.000 filas).

**Resultado:**
- **5.000 pedidos**, **12 columnas**, del 01/01/2024 al 31/12/2025.
- 100 % de valores válidos: 0 errores, 0 vacíos y 0 duplicados.
- Tipos detectados correctamente: enteros (ID_Pedido, Unidades_Vendidas, Precio_Unitario, Ingresos), decimal (Costo), fecha (Fecha_Pedido) y texto (Estación, ID_Cliente, Segmento_Cliente, Región, País, Categoría_Producto).
- **Columnas clave:** `Ingresos` y `Costo` (rentabilidad), `Fecha_Pedido` (tiempo), `País`, `Segmento_Cliente` y `Categoría_Producto` (comparaciones).

## 🔹 Paso 2: Preparación de los datos

**Objetivo:** fecha en formato español (Latinoamérica), tipos numéricos correctos, columna condicional `Nivel_Venta` y validación de calidad.

**Lo que hice en Power Query:**

1. **Fecha en español (Latinoamérica):** *Change Type → Using Locale → Date → Spanish (Ecuador)*. Así el formato queda guardado en el archivo y no depende de la configuración de Windows de quien lo abra.
2. **Tipos numéricos:** validados sin necesidad de correcciones (enteros para unidades, precio e ingresos; decimal para costo).
3. **Columna condicional `Nivel_Venta`** (*Add Column → Conditional Column*):
   - Si `Ingresos` es **mayor o igual** a 1000 → `"Venta Alta"`
   - En otro caso → `"Venta Baja"`
   - Tipo de dato: texto.
4. **Validación con Column profile:** 13 columnas con 100 % de valores válidos.

**Resultado:**

| Nivel_Venta | Pedidos |
| --- | --- |
| Venta Baja | 2.792 |
| Venta Alta | 2.208 |

**Prueba del caso límite:** hay 3 pedidos con ingresos exactamente de 1.000 (ID 87, 99 y 4832), y los tres quedaron como *Venta Alta*. Eso confirma que el operador es "mayor o igual" y no "mayor que".

## 🔹 Paso 3: Diseño y planificación del dashboard

#### Vista Overview - ¿Cómo ha evolucionado el ingreso total entre 2024 y 2025?
KPIs (4):
- Ingresos totales: la métrica principal de desempeño comercial.
- Ganancia (Ingresos - Costo): muestra si las ventas son rentables, no solo cuánto se vende.
- Margen %: permite comparar la rentabilidad entre años y segmentos aunque cambie el volumen.
- Pedidos: indica si los cambios en ingresos vienen de vender más o menos veces.

Gráficos:
- Gráfico de tiempo: líneas de Ingresos por mes, una línea por año (2024 en gris, 2025 en azul). Muestra la tendencia y los patrones estacionales, y permite comparar mes a mes.
- Comparación geográfica: columnas agrupadas de Ingresos por País y Año. Con solo 3 países, las columnas comparan mejor que un mapa.
- Comparación de clientes: columnas de Ingresos por Segmento de cliente, ordenadas de mayor a menor, para ver qué segmentos sostienen el negocio.

Jerarquía visual:
- Título arriba a la izquierda y filtro de Año arriba a la derecha.
- Fila de 4 KPIs debajo del título: es lo primero que lee el directivo.
- Gráfico de líneas a todo el ancho, en el centro: es el gráfico principal.
- Abajo, dos gráficos lado a lado (País y Segmento).
- Cada gráfico tiene un título que dice la conclusión, no solo el tema.

Filtro(s):
- Año (sincronizado con la Vista Detalle)

#### Vista Detalle - ¿Por qué han caído los ingresos?
Gráficos:
- Comparación estacional: columnas agrupadas - `Ingresos` vs `Estación`, por Año. Muestra en qué temporada se concentra la caída.
- Comparación de segmentos: barras horizontales agrupadas - `Ingresos` vs `Segmento_Cliente`, por Año. Identifica qué tipo de cliente explica la variación.
- Comparación de categorías: columnas agrupadas - `Ingresos` vs `Categoría_Producto`, por Año. Muestra si la caída es de un producto en particular o general.
- Tabla por Segmento y Categoría con 7 columnas:
  - Conteo `ID_Pedido` (Pedidos)
  - Conteo único `ID_Cliente` (Clientes únicos)
  - Segmento y Categoría
  - Ingresos
  - Ganancia
  - Margen %

Filtros (mínimo 2):
- `Fecha_Pedido` (Año, sincronizado con la Vista General)
- `País`
- `Segmento_Cliente`

Interacciones:
- Al hacer clic en un segmento, estación o categoría, los demás gráficos se filtran (no solo se resaltan), para explorar las causas con un clic.

## 🔹 Paso 4: Construcción del dashboard

### Modelo de datos y medidas DAX

Renombré la tabla a `Ventas` para que las fórmulas fueran cortas y legibles, y creé una columna para separar los años:

```DAX
Año = YEAR(Ventas[Fecha_Pedido])
```

Medidas usadas en todo el dashboard:

```DAX
Ingresos totales = SUM(Ventas[Ingresos])
Costo total      = SUM(Ventas[Costo])
Ganancia         = [Ingresos totales] - [Costo total]
Margen %         = DIVIDE([Ganancia], [Ingresos totales])
Pedidos          = DISTINCTCOUNT(Ventas[ID_Pedido])
Clientes         = DISTINCTCOUNT(Ventas[ID_Cliente])
Pedidos (texto)  = FORMAT([Pedidos], "#,0")
```

**Verificación de las medidas** (comparadas con un cálculo independiente en pandas):

| Año | Ingresos | Ganancia | Margen | Pedidos | Clientes |
| --- | --- | --- | --- | --- | --- |
| 2024 | 2.863.384 | 1.005.812 | 35,1 % | 2.550 | 2.234 |
| 2025 | 2.668.610 | 935.938 | 35,1 % | 2.450 | 2.117 |
| **Total** | **5.531.994** | **1.941.749** | **35,1 %** | **5.000** | **3.821** |

El total de clientes no es la suma de los dos años porque `DISTINCTCOUNT` cuenta una sola vez a quienes compraron en ambos.

### Vistas construidas

**Vista General:** 4 tarjetas KPI, línea de ingresos mensuales 2024 vs 2025, columnas por país y año, y columnas por segmento.

**Vista Detalle:** ingresos por estación, segmento y categoría (2024 vs 2025) y tabla por segmento y categoría con pedidos, clientes únicos, ingresos, ganancia y margen.

### Decisiones de diseño
- Títulos que comunican la conclusión de cada gráfico.
- Colores iguales en todo el dashboard: gris `#B3B3B3` para 2024 y azul `#1F4E79` para 2025.
- Etiquetas de datos en las barras; leyenda arriba a la izquierda; sin subtítulos automáticos.
- Líneas rectas (interpolación lineal) para no sugerir variaciones que no están en los datos.

### Filtros e interacciones
- Segmentador de **Año** sincronizado entre las dos páginas.
- Segmentadores de **País** y **Segmento** en la Vista Detalle.
- *Edit interactions*: un clic en un segmento **filtra** los gráficos de estación y categoría.

## 🔹 Paso 5: Narrativa con modelo SCQA

#### 🖥️ Vista General (Overview)

**S (Situación):**
Andes Retail Group vende en Perú, Chile y Colombia. Entre 2024 y 2025 generó 5,53M en ingresos y 1,94M de ganancia, con un margen de 35,1 % y 5.000 pedidos.

**C (Complicación):**
En 2025 los ingresos cayeron 6,8 % (de 2,86M a 2,67M, unos −195 mil). La caída no es pareja durante el año: se concentra en los meses de verano (diciembre a febrero). Perú es el país que más cayó (−10,4 %).

**Q (Pregunta):**
¿La caída es un problema de rentabilidad, de un mercado específico o de un tipo de cliente y temporada?

**A (Respuesta):**
No es un problema de rentabilidad: el margen se mantuvo en 35,1 % ambos años. Es un problema de volumen en la temporada alta. El verano concentra el 40 % de los ingresos del negocio, y ahí es donde 2025 quedó por debajo de 2024.

---

#### 🔎 Vista Detalle

**S (Situación):**
El verano es la temporada más importante: genera 2,24M de los 5,53M y sus pedidos son mucho más grandes (unas 18 unidades por pedido, frente a 5 en invierno).

**C (Complicación):**
Los ingresos de verano cayeron 203 mil (−17 %), más que la caída total del año. Por segmento, Premium bajó 196 mil (−14 %), mientras Estándar se mantuvo estable y Económico creció.

**Q (Pregunta):**
¿Qué explica la caída: los productos, los precios o los clientes?

**A (Respuesta):**
La caída viene de los clientes Premium en verano: sus ingresos en esa temporada bajaron de 0,61M a 0,44M (−168 mil). No es un problema de producto, porque la caída es parecida en Deportes, Electrónica y Ropa, y Hogar incluso creció (+3 %). Tampoco es de precios ni de costos, porque el margen se mantiene cerca de 35 % en todas las combinaciones de segmento y categoría.

Recomendación: lanzar una campaña de retención y reactivación de clientes Premium antes de la temporada alta (noviembre), priorizando Chile y Perú, donde Premium cayó más; e investigar por qué estos clientes compraron menos en el verano 2025 (competencia, disponibilidad de stock o campañas).

### 💬 Mensaje ejecutivo para Slack

**📊 Actualización desempeño comercial**

Equipo, resumen del análisis 2024–2025:

- Los ingresos de 2025 cayeron 6,8 % (−195 mil), pero el margen se mantuvo en 35,1 %: no es un problema de rentabilidad.
- La caída se concentra en el verano (dic–feb), nuestra temporada más fuerte (40 % de los ingresos).
- El origen son los clientes Premium: en verano compraron 168 mil menos que en 2024. Estándar está estable y Económico crece.
- Por categoría la caída es pareja; Hogar es la única que crece (+3 %).

👉 Propuesta: campaña de retención Premium antes del próximo verano (noviembre), priorizando Chile y Perú.

El dashboard está disponible para explorar por país, segmento y año.

## ✅ Conclusión

La caída de ingresos de 2025 (−6,8 %) **no es un problema de rentabilidad** (margen estable de 35,1 %) ni **de producto** (la caída es pareja entre categorías). Se explica por **menos compras de clientes Premium en la temporada de verano**. La acción recomendada es una campaña de retención Premium antes de noviembre, priorizando Chile y Perú.

**Archivos del proyecto:** el dashboard está en `dashboard/proyecto9_stefan_loosveld.pbix` y el dataset en `data/Andes_Retail_Group_2024_2025.xlsx`.